# Machine Sound Anomaly Detection: DCASE 2020 Task 2 (Autoencoder)

**Setup on Kaggle**
1. Add the DCASE 2020 Task 2 dev dataset via **Add Input** (right sidebar).
2. Turn on a GPU: Settings > Accelerator > GPU (optional, CPU also works).
3. Run all cells.

Method: one autoencoder per machine ID, trained on **normal sounds only**. Anomaly score = reconstruction error. Evaluated with AUC and pAUC (max FPR 0.1).

In [1]:
import os, glob, random, re, time
import numpy as np, pandas as pd
import librosa
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------
DATA_ROOT     = "/kaggle/input"      # searched recursively
MACHINE_TYPES = None                 # None = all found, or e.g. ["fan", "pump"] for a quick run
SR_MEL = dict(n_fft=1024, hop_length=512, n_mels=64, power=2.0)
FRAMES   = 5                         # context frames stacked into one input vector
EPOCHS   = 50
BATCH    = 512
LR       = 1e-3
OUT_DIR  = "/kaggle/working/models"
SEED     = 42
# ----------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Find the dataset
Auto-detects `train/` and `test/` folders wherever the dataset is mounted.

In [2]:
train_dirs = sorted({os.path.dirname(p) for p in glob.glob(f"{DATA_ROOT}/**/train/*.wav", recursive=True)})
machines = {}
for td in train_dirs:
    mtype = os.path.basename(os.path.dirname(td))
    machines[mtype] = dict(train=td, test=os.path.join(os.path.dirname(td), "test"))

if MACHINE_TYPES:
    machines = {k: v for k, v in machines.items() if k in MACHINE_TYPES}

assert machines, "No dataset found. Check that you added the DCASE 2020 Task 2 dataset under Add Input."
for k, v in machines.items():
    print(k, "->", v["train"])

ToyCar -> /kaggle/input/datasets/daisukelab/dc2020task2/ToyCar/train
ToyConveyor -> /kaggle/input/datasets/daisukelab/dc2020task2/ToyConveyor/train
fan -> /kaggle/input/datasets/daisukelab/dc2020task2/fan/train
pump -> /kaggle/input/datasets/daisukelab/dc2020task2/pump/train
slider -> /kaggle/input/datasets/daisukelab/dc2020task2/slider/train
valve -> /kaggle/input/datasets/daisukelab/dc2020task2/valve/train


## 2. Feature extraction (log-mel, stacked frames)

In [4]:
def file_to_logmel(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, **SR_MEL)
    return 20.0 / SR_MEL["power"] * np.log10(mel + np.finfo(float).eps)   # (n_mels, T)

def stack_frames(logmel, frames=FRAMES):
    n_mels, T = logmel.shape
    n_vec = T - frames + 1
    if n_vec < 1:
        return np.empty((0, n_mels * frames), dtype=np.float32)
    vec = np.zeros((n_vec, n_mels * frames), dtype=np.float32)
    for t in range(frames):
        vec[:, n_mels * t: n_mels * (t + 1)] = logmel[:, t: t + n_vec].T
    return vec

def files_to_vectors(files):
    return np.concatenate([stack_frames(file_to_logmel(f)) for f in files], axis=0)

def machine_id(fname):
    m = re.search(r"id_(\d+)", os.path.basename(fname))
    return m.group(1) if m else "00"

def label_of(fname):
    return 1 if os.path.basename(fname).startswith("anomaly") else 0

## 3. Model
Fully connected autoencoder (same architecture as the official DCASE baseline).

In [5]:
class AutoEncoder(nn.Module):
    def __init__(self, dim, hidden=128, bottleneck=8):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Linear(i, o), nn.BatchNorm1d(o), nn.ReLU())
        self.enc = nn.Sequential(blk(dim, hidden), blk(hidden, hidden), blk(hidden, hidden), blk(hidden, hidden), blk(hidden, bottleneck))
        self.dec = nn.Sequential(blk(bottleneck, hidden), blk(hidden, hidden), blk(hidden, hidden), blk(hidden, hidden), nn.Linear(hidden, dim))
    def forward(self, x):
        return self.dec(self.enc(x))

def train_model(X, epochs=EPOCHS):
    dim = X.shape[1]
    mu, sd = X.mean(0, keepdims=True), X.std(0, keepdims=True) + 1e-6      # normalize using normal data only
    Xn = torch.tensor((X - mu) / sd, dtype=torch.float32)
    idx = torch.randperm(len(Xn)); n_val = int(0.1 * len(Xn))
    val, tr = Xn[idx[:n_val]], Xn[idx[n_val:]]
    model = AutoEncoder(dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    best, best_state = 1e9, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr))
        for i in range(0, len(tr), BATCH):
            b = tr[perm[i:i + BATCH]].to(device)
            if len(b) < 2: continue
            opt.zero_grad(); loss = loss_fn(model(b), b); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = np.mean([loss_fn(model(val[i:i+4096].to(device)), val[i:i+4096].to(device)).item()
                          for i in range(0, len(val), 4096)])
        if vl < best:
            best, best_state = vl, {k: v.detach().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 10 == 0:
            print(f"    epoch {ep+1:3d}  val_loss {vl:.4f}")
    model.load_state_dict(best_state); model.eval()
    return model, mu, sd

@torch.no_grad()
def anomaly_score(model, mu, sd, path):
    v = torch.tensor((stack_frames(file_to_logmel(path)) - mu) / sd, dtype=torch.float32).to(device)
    return float(((model(v) - v) ** 2).mean().item())

## 4. Train and evaluate per machine ID

In [6]:
results = []
for mtype, paths in machines.items():
    train_files = sorted(glob.glob(os.path.join(paths["train"], "*.wav")))
    test_files  = sorted(glob.glob(os.path.join(paths["test"],  "*.wav")))
    ids = sorted({machine_id(f) for f in train_files})
    print(f"\n=== {mtype}: {len(train_files)} train files, {len(test_files)} test files, ids={ids}")

    for mid in ids:
        t0 = time.time()
        tr = [f for f in train_files if machine_id(f) == mid]
        te = [f for f in test_files  if machine_id(f) == mid]
        print(f"  [id_{mid}] extracting features from {len(tr)} normal files ...")
        X = files_to_vectors(tqdm(tr, leave=False))
        print(f"  [id_{mid}] training on {X.shape[0]:,} vectors ...")
        model, mu, sd = train_model(X)
        torch.save({"state": model.state_dict(), "mu": mu, "sd": sd}, f"{OUT_DIR}/{mtype}_id_{mid}.pt")

        if te:
            y_true = [label_of(f) for f in te]
            y_score = [anomaly_score(model, mu, sd, f) for f in tqdm(te, leave=False)]
            auc  = roc_auc_score(y_true, y_score)
            pauc = roc_auc_score(y_true, y_score, max_fpr=0.1)
            print(f"  [id_{mid}] AUC={auc:.4f}  pAUC={pauc:.4f}  ({time.time()-t0:.0f}s)")
            results.append(dict(machine=mtype, id=mid, AUC=auc, pAUC=pauc))

df = pd.DataFrame(results)
df.to_csv("/kaggle/working/results.csv", index=False)
df


=== ToyCar: 4000 train files, 2459 test files, ids=['01', '02', '03', '04']
  [id_01] extracting features from 1000 normal files ...


  0%|          | 0/1000 [00:00<?, ?it/s]

  [id_01] training on 340,000 vectors ...
    epoch  10  val_loss 0.3172
    epoch  20  val_loss 0.3127
    epoch  30  val_loss 0.3093
    epoch  40  val_loss 0.3075
    epoch  50  val_loss 0.3063


  0%|          | 0/614 [00:00<?, ?it/s]

  [id_01] AUC=0.8067  pAUC=0.6835  (294s)
  [id_02] extracting features from 1000 normal files ...


...

  0%|          | 0/892 [00:00<?, ?it/s]

  [id_06] training on 275,628 vectors ...
    epoch  10  val_loss 0.2533
    epoch  20  val_loss 0.2473
    epoch  30  val_loss 0.2431
    epoch  40  val_loss 0.2419
    epoch  50  val_loss 0.2404


  0%|          | 0/220 [00:00<?, ?it/s]

  [id_06] AUC=0.5927  pAUC=0.4829  (265s)


,machine,id,AUC,pAUC
0,ToyCar,01,0.806677,0.683527
1,ToyCar,02,0.875871,0.778522
2,ToyCar,03,0.699515,0.579118
3,ToyCar,04,0.886038,0.734147
4,ToyConveyor,01,0.748856,0.602187
5,ToyConveyor,02,0.646697,0.560953
6,ToyConveyor,03,0.727452,0.586511
7,fan,00,0.550246,0.496314
8,fan,02,0.772786,0.607682
9,fan,04,0.583017,0.518905


In [7]:
if len(df):
    print(df.groupby("machine")[["AUC", "pAUC"]].mean().round(4))
    print("\nOverall mean AUC :", round(df.AUC.mean(), 4))
    print("Overall mean pAUC:", round(df.pAUC.mean(), 4))

                AUC    pAUC
machine                    
ToyCar       0.8170  0.6938
ToyConveyor  0.7077  0.5832
fan          0.6889  0.5711
pump         0.7602  0.6652
slider       0.8206  0.6244
valve        0.6635  0.5091

Overall mean AUC : 0.7445
Overall mean pAUC: 0.6089


In [8]:
import shutil
shutil.make_archive("/kaggle/working/models_zip", "zip", "/kaggle/working/models")

'/kaggle/working/models_zip.zip'

In [1]:
import glob, requests
from sklearn.metrics import roc_auc_score

URL = "https://sound-guard-fbfubr8tr-harry-9194.vercel.app/api/analyze-audio?machine=pump&machineId=04"

normal  = sorted(glob.glob("/kaggle/input/**/pump/test/normal_id_04_*.wav",  recursive=True))[:50]
anomaly = sorted(glob.glob("/kaggle/input/**/pump/test/anomaly_id_04_*.wav", recursive=True))[:50]
print(len(normal), "normal files,", len(anomaly), "anomaly files")

labels, scores, threshold = [], [], None
for label, files in ((0, normal), (1, anomaly)):
    for i, f in enumerate(files):
        r = requests.post(URL, data=open(f, "rb").read(),
                          headers={"Content-Type": "audio/wav"}, timeout=60).json()
        if "error" in r:
            raise SystemExit(f"{f}: {r['error']}")
        labels.append(label); scores.append(r["score"]); threshold = r["threshold"]
        if (i + 1) % 10 == 0:
            print("done", i + 1, "of", len(files), "(label", label, ")")

tp = sum(1 for l, s in zip(labels, scores) if l == 1 and s > threshold)
fp = sum(1 for l, s in zip(labels, scores) if l == 0 and s > threshold)
print("\nAUC =", round(roc_auc_score(labels, scores), 4))
print("Recall (faults caught) =", f"{tp / labels.count(1):.0%}")
print("False alarms on normal =", f"{fp / labels.count(0):.0%}")

50 normal files, 50 anomaly files
done 10 of 50 (label 0 )
done 20 of 50 (label 0 )
done 30 of 50 (label 0 )
done 40 of 50 (label 0 )
done 50 of 50 (label 0 )
done 10 of 50 (label 1 )
done 20 of 50 (label 1 )
done 30 of 50 (label 1 )
done 40 of 50 (label 1 )
done 50 of 50 (label 1 )

AUC = 0.9752
Recall (faults caught) = 98%
False alarms on normal = 14%


In [2]:
import glob, requests
from sklearn.metrics import roc_auc_score
URL = "https://sound-guard-fbfubr8tr-harry-9194.vercel.app/api/analyze-audio?machine=pump&machineId=04"
normal  = sorted(glob.glob("/kaggle/input/**/pump/test/normal_id_04_*.wav",  recursive=True))[50:]
anomaly = sorted(glob.glob("/kaggle/input/**/pump/test/anomaly_id_04_*.wav", recursive=True))[50:]
print(len(normal), "fresh normal,", len(anomaly), "fresh anomaly")
labels, scores, thr = [], [], None
for label, files in ((0, normal), (1, anomaly)):
    for f in files:
        r = requests.post(URL, data=open(f, "rb").read(), headers={"Content-Type": "audio/wav"}, timeout=60).json()
        labels.append(label); scores.append(r["score"]); thr = r["threshold"]
tp = sum(l == 1 and s > thr for l, s in zip(labels, scores)); fp = sum(l == 0 and s > thr for l, s in zip(labels, scores))
print("AUC", round(roc_auc_score(labels, scores), 4), "| recall", f"{tp/labels.count(1):.0%}", "| false alarms", f"{fp/labels.count(0):.0%}")

50 fresh normal, 50 fresh anomaly
AUC 0.9764 | recall 98% | false alarms 14%
